2PC tackelt het probleem van transacties van een database dat gedistribueerd is over meerder servers of schijven. Problemen die komen kijken bij deze casus

- **Site failures**  
  Een site is operationeel of uitgevallen (fail/stop-gedrag).

- **Partial failure in the network**  
  Sommige sites zijn operationeel, terwijl andere sites uitgevallen zijn.

- **Total failure**  
  Alle sites zijn uitgevallen.

- **Communication failure**  
  Netwerkpartities of componenten die niet met elkaar kunnen communiceren.

#### Aannames

- **Undeliverable messages dropped**  
  Berichten die niet kunnen worden afgeleverd, worden verworpen.

- **Failure detection by timeout**  
  Fouten worden gedetecteerd via een timeout-mechanisme.

De twee fase commit protocol werkt als volgt, je hebt een COORDinator, dit is de source van de transactie. Denk aan een ING entiteit in Amsterdam dat een transactie wil uitvoeren in een database die in meerdere panden in het land gestationeerd zijn. Deze COORD vraag aan de PARTicipants, de andere fysiek aanwezige disks, sites, nodes of een andere database instance met betrekking tot dezelfde database, of ze deze transactie kunnen committen. Samengevat is de filosofie rond 2PC dan ook:

$$
    \text{Either all commit or all abort}
$$

D.w.z. samen thuis samen uit.

**Fase 1**

De COORD stuurt naar alle PART's: $\text{[prepare-T]}$, elke PART checkt dan of haar database volledig/valide is of zij kan committen en of zij resources kan locken. Mocht dit het geval zijn zal de PART het volgende terug sturen $\text{[ready T]}$, zo niet $\text{[abort T]}$. Waarbij $\text{T}$ symbolisch staat voor de transactie.

**Fase 2**

Na het voting proces zal de COORD, in het geval dat alle PART's $\text{[ready T]}$ hebben gestemd, een global commit uitvoeren $\text{GLOBAL COMMIT}$. Hierna zullen alle PART's en de COORD de transactie committen volgens de besproken methoden van een op zich zelf staande transactie. Zo niet zal de COORD een global abort uitvoeren $\text{GLOBAL ABORT}$, waarna alles PART's en de COORD een rollback zullen doen. Dit houdt in dat de transactie dus niet wordt uitgevoerd.

**DT-log**

*Fase 1*

Elke PART's (hier zit de COORD bij in) houden een DT-log bij, dit lijkt op de log die we kunnen uit recovery. Waneer de COORD aan fase 1 begint zal hij in zijn DT-log $\langle \text{prepare-T} \rangle$ zetten. 

 Wanneer de PART's de $\text{[prepare-T]}$ ontvangen, zullen zij $\langle \text{abort-T} \rangle$ loggen als zij abort stemmen en uiteraard $\langle \text{ready-T} \rangle$ loggen wanneer zij ready stemmen.

 *Fase 2*

 Wanneer de COORD alle stemmen verzameld en elke stem (inclusief die van de COORD) is positief zal de COORD $\langle \text{Commit-T} \rangle$ loggen. Is er een stem (of meer) die negatief zijn zal de COORD $\langle \text{Abort-T} \rangle$ loggen en alleen naar de positieve stemmen $\text{[Abort-T]}$ sturen.

**Wanneer gaat het fout?**

